# Rancher AI from a notebook

The same profiles as the Profiles page: list them, start a training run, follow it, and call an inference endpoint. `Client()` connects with your kubeconfig context (or `RANCHER_URL` + `RANCHER_TOKEN`, or the pod's service account when this notebook runs in the cluster), so it can do exactly what you can do in the Rancher UI.

## 1. Connect

In [ ]:
from rancher_ai import Client

ai = Client(project="team-a")   # project = your namespace
ai.whoami()

## 2. Explore available profiles

In [ ]:
df = ai.profiles.list()
df

In [ ]:
ai.profiles.get("shared-gpu-dev").editable   # what this profile lets you set

## 3. Submit a training run

`shared-gpu-dev` gives one worker 4 GiB of the project's shared GPU, beside whatever else is using it. Its runs are named `dev-…`; the profile fixes the image, resources and GPU, and you supply the code.

In [ ]:
TRAIN = """
import time, torch
free, total = torch.cuda.mem_get_info()
print(f"GPU {torch.cuda.get_device_name(0)}: {free/2**30:.2f} GiB free of {total/2**30:.2f} GiB", flush=True)
model = torch.nn.Sequential(torch.nn.Linear(1024, 2048), torch.nn.ReLU(), torch.nn.Linear(2048, 10)).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
for epoch in range(1, 6):
    for _ in range(200):
        x, y = torch.randn(256, 1024, device="cuda"), torch.randint(0, 10, (256,), device="cuda")
        loss = torch.nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"INFO | Epoch {epoch}/5 | loss={loss.item():.3f}", flush=True)
    time.sleep(2)
"""

run = ai.runs.create(profile="shared-gpu-dev", name="dev-notebook-demo", script=TRAIN, runtime_hours=1)
run

## 4. Monitor the run

In [ ]:
run.wait()      # blocks until Completed or Failed; run.wait({"Running"}) to stop earlier
run.status()

In [ ]:
run.logs(tail=10)

In [ ]:
ai.runs.table()   # everything in the project, as on the Deployments page

## 5. Deploy an inference endpoint

An inference profile deploys a SUSE AI Factory blueprint. A project runs one workload per blueprint, so this reuses the endpoint if it is already there.

In [ ]:
try:
    endpoint = ai.endpoints.get("qwen-shared")
except LookupError:
    endpoint = ai.endpoints.create(profile="suse-inference-endpoint-qwen-shared", name="qwen-shared")
endpoint.wait()   # the first start downloads the model: a few minutes
endpoint

In [ ]:
endpoint.chat("What is a large language model?")

## 6. Clean up

In [ ]:
run.delete()   # helm uninstall; endpoint.delete() removes the endpoint